[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Async Queries &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell and the second defines the `Item` model and fills it.
Run both first, then any task in any order.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pydantic
import pymongo
from beanie import Document, init_beanie
from beanie.operators import And, GT, In, Or, RegEx
from pymongo import AsyncMongoClient

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def first_problem(error):
    """One line out of a pydantic ValidationError, which is otherwise several paragraphs."""
    problem = error.errors()[0]
    return f"{'.'.join(str(part) for part in problem['loc'])}: {problem['msg']}"


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


In [2]:
class Item(Document):
    sku: str
    kind: str
    price: float
    stock: int = 0

    class Settings:
        name = "queried"


client = AsyncMongoClient(URI)
await init_beanie(database=client.get_default_database(), document_models=[Item])
await Item.delete_all()
await Item.insert_many([
    Item(sku=f"S-{number}", kind=["tool", "part"][number % 2],
         price=10.0 * number, stock=number)
    for number in range(6)
])
print("documents:", await Item.find_all().count())


documents: 6


**1.** Two conditions.


In [3]:
found = await Item.find(Item.kind == "tool", Item.price > 10).to_list()
print([(item.sku, item.price) for item in found])


[('S-2', 20.0), ('S-4', 40.0)]


Two arguments to `find` are combined with and. Writing them as two chained `find` calls would mean
exactly the same thing.


**2.** Either of two kinds.


In [4]:
both = await Item.find(In(Item.kind, ["tool", "part"])).count()
one = await Item.find(In(Item.kind, ["tool"])).count()
print("either kind:", both, "| just tools:", one)


either kind: 6 | just tools: 3


`In` is a function because Python has no operator for "is one of these". It is the same filter
PyMongo would write as `{"kind": {"$in": [...]}}`.


**3.** The two dearest.


In [5]:
for item in await Item.find_all().sort(-Item.price).limit(2).to_list():
    print(f"  {item.sku}  {item.price:6.2f}")


  S-5   50.00
  S-4   40.00


`-Item.price` is descending. The sort field is checked against the model, so a typo here fails the
same way a typo in a filter does.


**4.** Nothing, three ways.


In [6]:
query = Item.find(Item.price > 10_000)

print("to_list():      ", await query.to_list())
print("first_or_none():", await query.first_or_none())
print("count():        ", await query.count())


to_list():       []
first_or_none(): None
count():         0


An empty list, `None` and zero. Only the first of those turns into an `IndexError` if the next line
assumes there was something.


**5.** One field only.


In [7]:
class SkuOnly(pydantic.BaseModel):
    sku: str


rows = await Item.find(Item.kind == "part").project(SkuOnly).to_list()
print("type:", type(rows[0]).__name__, "| values:", [row.sku for row in rows])


type: SkuOnly | values: ['S-1', 'S-3', 'S-5']


The documents crossing the network carry one field. What comes back is a `SkuOnly`, not an `Item`,
so anything downstream should say so.


**6.** A grouped report, typed.


In [8]:
class ByKind(pydantic.BaseModel):
    kind: str
    items: int
    stock: int


pipeline = [
    {"$group": {"_id": "$kind", "items": {"$sum": 1}, "stock": {"$sum": "$stock"}}},
    {"$project": {"_id": 0, "kind": "$_id", "items": 1, "stock": 1}},
    {"$sort": {"kind": 1}},
]

for row in await Item.aggregate(pipeline, projection_model=ByKind).to_list():
    print(f"  {row.kind:5} {row.items} items, {row.stock} in stock")

await Item.delete_all()
await client.close()


  part  3 items, 9 in stock
  tool  3 items, 6 in stock


The `$project` stage renames `_id` to `kind` because the model has a field called `kind`. The
pipeline's output names and the model's field names are a contract written in two places, and
Pydantic checks it.


---

&#8592; **Back to:** [Async Queries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/12-async-queries.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
